# GeoGovGadget — parcel boundary segmentation (lite prototype)

**Task:** raw satellite/drone image -> closed boundary-line mask (green line = clean parcel, per your labeling; overlap coloring is downstream Turf.js logic already in the app, not part of this model).

**Data:** ~60 paired images (`raw` + `plotted`) from your Drive. Only 60 pairs -> we use transfer learning (ImageNet-pretrained MobileNetV2 encoder) so the model converges fast without needing thousands of examples. Treat today's output as a lite prototype, not production accuracy — more data / an A100 later is the path to real accuracy.

**Folder convention expected in your Drive** (edit `DRIVE_ROOT` below to match):
```
DRIVE_ROOT/
  raw/       001.jpg, 002.jpg, ...       (unmarked satellite/drone images)
  plotted/   001.jpg, 002.jpg, ...       (same filenames, with red/green boundary lines drawn)
```
Filenames in `raw/` and `plotted/` must match 1:1 (same basename). If yours don't, rename them first — the loader below will error out loudly (not silently mismatch pairs) if it can't match everything.

In [ ]:
USE_DRIVE = False  # True = read from Google Drive (real run). False = read from local Colab session storage (quick smoke test).

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/GeoGovGadget_data'
    RAW_DIR = f'{DRIVE_ROOT}/raw'
    PLOTTED_DIR = f'{DRIVE_ROOT}/plotted'
    OUT_DIR = f'{DRIVE_ROOT}/outputs'
else:
    # For a smoke test: use the Colab file browser (folder icon in the left sidebar) to create
    # /content/raw and /content/plotted, then drag your 3 raw images into /content/raw and the
    # 3 matching plotted images into /content/plotted, using the SAME filenames in both
    # (e.g. raw/1.jpg <-> plotted/1.jpg). Files here vanish when the runtime resets — that's fine
    # for a smoke test, just re-upload before the real run.
    RAW_DIR = '/content/raw'
    PLOTTED_DIR = '/content/plotted'
    OUT_DIR = '/content/outputs'

MASK_DIR = f'{OUT_DIR}/masks'
MODEL_DIR = f'{OUT_DIR}/models'

import os
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PLOTTED_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print('RAW_DIR:', RAW_DIR)
print('PLOTTED_DIR:', PLOTTED_DIR)
print('Now upload your images into these two folders via the Colab file browser (left sidebar), then continue to the next cell.')

In [ ]:
!pip -q install opencv-python-headless albumentations tensorflow

## 1. Verify pairs match

In [ ]:
raw_files = sorted(os.listdir(RAW_DIR))
plotted_files = sorted(os.listdir(PLOTTED_DIR))

raw_basenames = {os.path.splitext(f)[0]: f for f in raw_files}
plotted_basenames = {os.path.splitext(f)[0]: f for f in plotted_files}

common = sorted(set(raw_basenames) & set(plotted_basenames))
missing_in_plotted = sorted(set(raw_basenames) - set(plotted_basenames))
missing_in_raw = sorted(set(plotted_basenames) - set(raw_basenames))

print(f'{len(common)} matched pairs')
if missing_in_plotted:
    print('Raw images with NO plotted counterpart (excluded):', missing_in_plotted)
if missing_in_raw:
    print('Plotted images with NO raw counterpart (excluded):', missing_in_raw)

MIN_PAIRS = 2 if not USE_DRIVE else 20  # smoke test just needs the pipeline to run, not to learn anything
assert len(common) >= MIN_PAIRS, f'Too few matched pairs ({len(common)}) to proceed — fix filenames in {RAW_DIR}/{PLOTTED_DIR} first.'
if len(common) < 20:
    print(f'NOTE: only {len(common)} pairs — this is a pipeline smoke test, not a real training run. '
          'Expect the model to memorize/do nothing useful. Re-run with USE_DRIVE=True and the full ~60 pairs for a real result.')

pairs = [(os.path.join(RAW_DIR, raw_basenames[k]), os.path.join(PLOTTED_DIR, plotted_basenames[k])) for k in common]

## 2. Auto-extract boundary masks from the plotted images

Since your lines are consistently red/green, we threshold them in HSV rather than hand-labeling — turns your 60 plotted images directly into training targets with zero extra annotation work.

In [ ]:
import cv2
import numpy as np

def extract_boundary_mask(plotted_bgr):
    hsv = cv2.cvtColor(plotted_bgr, cv2.COLOR_BGR2HSV)

    # red wraps around hue 0/180 -> two ranges
    red1 = cv2.inRange(hsv, (0, 80, 60), (10, 255, 255))
    red2 = cv2.inRange(hsv, (170, 80, 60), (180, 255, 255))
    green = cv2.inRange(hsv, (35, 60, 40), (90, 255, 255))

    mask = cv2.bitwise_or(cv2.bitwise_or(red1, red2), green)

    # close small gaps in hand-drawn lines so boundaries stay contiguous
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    return mask

mask_paths = []
for raw_path, plotted_path in pairs:
    plotted = cv2.imread(plotted_path)
    mask = extract_boundary_mask(plotted)
    out_path = os.path.join(MASK_DIR, os.path.basename(raw_path))
    cv2.imwrite(out_path, mask)
    mask_paths.append(out_path)

print(f'Wrote {len(mask_paths)} masks to {MASK_DIR}')

In [ ]:
# Sanity check a few pairs visually before training on garbage masks
import matplotlib.pyplot as plt

sample_idx = np.random.choice(len(pairs), size=min(3, len(pairs)), replace=False)
fig, axes = plt.subplots(len(sample_idx), 3, figsize=(12, 4 * len(sample_idx)))
if len(sample_idx) == 1:
    axes = axes[None, :]
for row, i in enumerate(sample_idx):
    raw = cv2.cvtColor(cv2.imread(pairs[i][0]), cv2.COLOR_BGR2RGB)
    plotted = cv2.cvtColor(cv2.imread(pairs[i][1]), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(mask_paths[i], cv2.IMREAD_GRAYSCALE)
    axes[row, 0].imshow(raw); axes[row, 0].set_title('raw'); axes[row, 0].axis('off')
    axes[row, 1].imshow(plotted); axes[row, 1].set_title('plotted (ground truth)'); axes[row, 1].axis('off')
    axes[row, 2].imshow(mask, cmap='gray'); axes[row, 2].set_title('extracted mask'); axes[row, 2].axis('off')
plt.tight_layout()
plt.show()

print('If the extracted mask does NOT clearly trace the boundary lines in the plotted image,')
print('stop here and widen/narrow the HSV ranges in extract_boundary_mask() above, then rerun.')

## 3. Data pipeline (heavy augmentation — only ~60 images)

In [ ]:
import albumentations as A
import tensorflow as tf

IMG_SIZE = 256

train_pairs, val_pairs = [], []
raw_paths = [p[0] for p in pairs]
rng = np.random.RandomState(42)
idx = rng.permutation(len(raw_paths))
n_val = max(1, min(len(raw_paths) - 1, int(round(0.2 * len(raw_paths))) if len(raw_paths) >= 6 else 1))
val_idx, train_idx = set(idx[:n_val].tolist()), set(idx[n_val:].tolist())

for i, raw_path in enumerate(raw_paths):
    mask_path = mask_paths[i]
    (train_pairs if i in train_idx else val_pairs).append((raw_path, mask_path))

print(f'train: {len(train_pairs)}  val: {len(val_pairs)}')

train_aug = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(rotate=(-20, 20), scale=(0.9, 1.1), translate_percent=(-0.05, 0.05), p=0.5),
    A.RandomBrightnessContrast(p=0.4),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.15),
])

val_aug = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE)])

def load_and_augment(raw_path, mask_path, augment):
    raw_path = raw_path.numpy().decode()
    mask_path = mask_path.numpy().decode()
    image = cv2.cvtColor(cv2.imread(raw_path), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    aug = train_aug if augment else val_aug
    result = aug(image=image, mask=mask)
    image, mask = result['image'], result['mask']

    image = image.astype('float32') / 255.0
    mask = (mask > 127).astype('float32')[..., None]
    return image, mask

def make_dataset(pair_list, augment, batch_size=8, shuffle=True, repeat=False):
    batch_size = min(batch_size, len(pair_list))
    raw_list = [p[0] for p in pair_list]
    mask_list = [p[1] for p in pair_list]
    ds = tf.data.Dataset.from_tensor_slices((raw_list, mask_list))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(pair_list))
    if repeat:
        ds = ds.repeat()
    def _map(raw_path, mask_path):
        image, mask = tf.py_function(
            lambda r, m: load_and_augment(r, m, augment),
            [raw_path, mask_path], [tf.float32, tf.float32])
        image.set_shape([IMG_SIZE, IMG_SIZE, 3])
        mask.set_shape([IMG_SIZE, IMG_SIZE, 1])
        return image, mask
    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

BATCH_SIZE = min(8, len(train_pairs))
# steps_per_epoch matters here since train_ds repeats indefinitely with augmentation
STEPS_PER_EPOCH = max(1, len(train_pairs) * 4 // BATCH_SIZE)  # ~4x augmented passes per epoch
train_ds = make_dataset(train_pairs, augment=True, batch_size=BATCH_SIZE, repeat=True)
val_ds = make_dataset(val_pairs, augment=False, batch_size=BATCH_SIZE, shuffle=False)

## 4. Model — U-Net with a pretrained MobileNetV2 encoder

Standard transfer-learning U-Net pattern: freeze the ImageNet-pretrained encoder, only train the decoder — this is what makes 48 training images workable in a couple hours instead of needing thousands.

In [ ]:
from tensorflow.keras import layers, models

def build_unet(img_size=IMG_SIZE):
    base = tf.keras.applications.MobileNetV2(input_shape=[img_size, img_size, 3], include_top=False)

    skip_names = [
        'block_1_expand_relu',   # 128x128
        'block_3_expand_relu',   # 64x64
        'block_6_expand_relu',   # 32x32
        'block_13_expand_relu',  # 16x16
        'block_16_project',      # 8x8 (bottleneck)
    ]
    skip_outputs = [base.get_layer(name).output for name in skip_names]
    encoder = models.Model(inputs=base.input, outputs=skip_outputs)
    encoder.trainable = False  # freeze for the lite prototype; unfreeze later for fine-tuning with more data

    inputs = layers.Input(shape=[img_size, img_size, 3])
    s1, s2, s3, s4, bottleneck = encoder(inputs)

    x = bottleneck
    for skip, filters in zip([s4, s3, s2, s1], [512, 256, 128, 64]):
        x = layers.Conv2DTranspose(filters, 3, strides=2, padding='same', activation='relu')(x)
        x = layers.Concatenate()([x, skip])
        x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)

    x = layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(x)
    outputs = layers.Conv2D(1, 1, padding='same', activation='sigmoid')(x)

    return models.Model(inputs, outputs, name='boundary_unet')

model = build_unet()
model.summary()

In [ ]:
import tensorflow.keras.backend as K

def dice_loss(y_true, y_pred, smooth=1.0):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def bce_dice_loss(y_true, y_pred):
    # boundary lines are a small fraction of pixels -> plain BCE alone underweights them
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return K.mean(bce) + dice_loss(y_true, y_pred)

def iou_metric(y_true, y_pred, threshold=0.5):
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)
    intersection = K.sum(y_true * y_pred_bin)
    union = K.sum(y_true) + K.sum(y_pred_bin) - intersection
    return (intersection + 1e-7) / (union + 1e-7)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=bce_dice_loss, metrics=[iou_metric])

In [ ]:
checkpoint_path = os.path.join(MODEL_DIR, 'boundary_unet_best.h5')

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor='val_iou_metric', mode='max', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_iou_metric', mode='max', patience=15, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_iou_metric', mode='max', factor=0.5, patience=6, verbose=1),
]

history = model.fit(
    train_ds,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_ds,
    epochs=80,
    callbacks=callbacks,
)

## 5. Inference — raw image -> closed boundary overlay

In [ ]:
def predict_boundary(raw_image_path, threshold=0.4):
    orig = cv2.cvtColor(cv2.imread(raw_image_path), cv2.COLOR_BGR2RGB)
    h, w = orig.shape[:2]

    resized = cv2.resize(orig, (IMG_SIZE, IMG_SIZE)).astype('float32') / 255.0
    pred = model.predict(resized[None, ...], verbose=0)[0, ..., 0]
    pred_mask = (pred > threshold).astype(np.uint8) * 255
    pred_mask = cv2.resize(pred_mask, (w, h), interpolation=cv2.INTER_NEAREST)

    # close gaps in the predicted line so contours form closed polygons
    kernel = np.ones((5, 5), np.uint8)
    closed = cv2.morphologyEx(pred_mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(closed, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    overlay = orig.copy()
    cv2.drawContours(overlay, contours, -1, (0, 255, 0), 2)  # green = drawn boundary; red/green overlap coloring happens downstream in the app, not here

    return overlay, pred_mask, contours

# quick look at a validation image
test_raw = val_pairs[0][0]
overlay, mask, contours = predict_boundary(test_raw)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(mask, cmap='gray'); axes[0].set_title('predicted mask'); axes[0].axis('off')
axes[1].imshow(overlay); axes[1].set_title(f'{len(contours)} closed boundaries found'); axes[1].axis('off')
plt.show()

## 6. Export

Saves the full model (for the demo) plus a TFLite conversion (matches the edge-device story in `PROJECT_PLAN.md` — offline on-device inference).

In [ ]:
final_path = os.path.join(MODEL_DIR, 'boundary_unet_final.h5')
model.save(final_path)
print('Saved:', final_path)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = os.path.join(MODEL_DIR, 'boundary_unet.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
print('Saved:', tflite_path)

## Next steps (after tonight)

- **Wire into the app:** `README.md` already documents the swap-in point — `generateFeatures()` in `lib/geo.js` is where `/api/segment`'s deterministic demo polygons get replaced with real model output. This model's `predict_boundary()` contours are the piece that plugs in there (vectorize with `cv2.findContours` -> `Shapely`/`Turf.js`-compatible GeoJSON).
- **More data / A100:** with only ~48 training images, expect rough edges on complex/irregular plots — this is honestly a lite prototype, matching `PROJECT_PLAN.md`'s framing. Unfreezing the MobileNetV2 encoder (`encoder.trainable = True`) and fine-tuning end-to-end with a lower learning rate is the natural next step once you have more images and a real GPU (A100), not for tonight.
- **Red/green overlap coloring** is not part of this model — it's the existing Turf.js polygon-intersection logic already in the app. Keep that as-is; this model only needs to output closed boundary polygons.